# DE.LAS — Exercício: berries na PokeAPI + PySpark

Complete as partes marcadas com **`# TODO`**. O notebook guiado usa a rota **`/pokemon`** e agrega por **tipo primário**; aqui você trabalha com **`/berry`** e agrega por **firmeza** com **média de tempo de crescimento** — outra rota da API e outro tipo de agregação.

Objetivos:

- Ingerir lista + detalhe via HTTP (padrão semelhante ao tutorial, outro recurso).
- Gravar bronze em JSONL e transformar com PySpark (`explode`, structs aninhados).
- Publicar a camada ouro em Parquet e em SQL; opcionalmente visualizar.


## Configuração do ambiente (Colab)

Execute as duas células abaixo sem alterações. No Colab, o diretório `/content` existe; localmente usamos `./content`.


In [ ]:
!pip install -q pyspark requests matplotlib pandas

In [ ]:
import os
from pathlib import Path

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DE_LAS_BerryExercise").master("local[*]").getOrCreate()

IS_COLAB = os.path.exists("/content")
DATA_DIR = Path("/content/delas_data") if IS_COLAB else Path("./content/delas_data")


## Exercício 1 — Ingestão

**Tarefa:** obter a lista de bagas (berries) com `GET /berry` e `limit=30`, depois, para cada item em `results`, fazer `GET` na `url` e acumular o JSON completo em `records`.

**Opcional:** `try`/`except` por requisição para não perder o lote inteiro se uma chamada falhar.


In [ ]:
import json
import time
import requests

BASE = "https://pokeapi.co/api/v2"
LIMIT = 30

session = requests.Session()
session.headers.update({"User-Agent": "DE.LAS-bootcamp-exercise/1.0"})

try:
    listing = session.get(f"{BASE}/berry", params={"limit": LIMIT}, timeout=60)
    listing.raise_for_status()
    results = listing.json()["results"]
except requests.exceptions.RequestException as e:
    print(f"Erro ao buscar lista de berries: {e}")
    results = []  # evita quebrar o restante

records = []

for item in results:
    try:
        detail = session.get(item["url"], timeout=60)
        detail.raise_for_status()
        records.append(detail.json())
    except requests.exceptions.RequestException as e:
        print(f"Erro ao buscar detalhe de {item['url']}: {e}")
        continue  # pula esse item e segue o loop

    time.sleep(0.05)

# validação mais segura
if len(records) != LIMIT:
    print(f"Atenção: esperado {LIMIT}, mas obteve {len(records)} registros")

if records:
    print(records[0]["name"], records[0]["growth_time"])

## Exercício 2 — Armazenamento (bronze)

**Tarefa:** gravar `records` em `DATA_DIR / "bronze" / "berry_raw.jsonl"` (uma linha JSON por registro de baga).


In [ ]:
RAW_PATH = DATA_DIR / "bronze" / "berry_raw.jsonl"
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)

# TODO: abrir RAW_PATH em escrita utf-8 e escrever json.dumps(registro) + quebra de linha para cada registro em records

assert RAW_PATH.exists(), "Arquivo bruto não foi criado"
RAW_PATH


## Exercício 3 — Transformação (PySpark)

**Tarefa:**

1. `raw_df = spark.read.json(RAW_PATH)`.
2. `berries_df`: colunas `id`, `name`, `growth_time`, `firmness_name` (utilize `F.col("firmness.name")`).
3. `flavors_df`: aplique `explode` em `flavors`; colunas `id`, `name`, `flavor_name` (via `fl.flavor.name`), `potency` (via `fl.potency`).

Mostre schema e algumas linhas de `flavors_df`.


In [ ]:
from pyspark.sql import functions as F

# TODO: definir raw_df = spark.read.json(...)
raw_df = None

# TODO: montar berries_df a partir de firmness.name como firmness_name
berries_df = None

# TODO: montar flavors_df com explode(flavors)
flavors_df = None

berries_df.printSchema()
flavors_df.show(8, truncate=False)


## Exercício 4 — Disponibilização (camada ouro)

**Tarefa:** agregar por `firmness_name` (nome da firmeza):

- `avg(growth_time)` como `avg_growth_time`
- contagem de linhas como `berry_count`

Ordene por `avg_growth_time` em ordem decrescente. Grave em Parquet em `DATA_DIR / "gold" / "berry_by_firmness"`, registre a visão temporária `berry_by_firmness` e execute um `SELECT` ordenado.

**Opcional:** gráfico de barras (`matplotlib`) com `avg_growth_time` por firmeza.

**Desafio SQL:** tipos de firmeza com `berry_count >= 2`.


In [ ]:
# TODO: gold_df = berries_df.groupBy("firmness_name").agg(...)  # agregações pedidas acima
gold_df = None

GOLD_DIR = DATA_DIR / "gold" / "berry_by_firmness"
GOLD_DIR.parent.mkdir(parents=True, exist_ok=True)

# TODO: gravar gold_df em Parquet com mode("overwrite") no caminho de GOLD_DIR
# TODO: registrar visão temporária berry_by_firmness a partir de gold_df

spark.sql("SELECT * FROM berry_by_firmness ORDER BY avg_growth_time DESC").show(truncate=False)
